In [1]:
import scipy.sparse as sp
import pandas as pd
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns


In [2]:
def marker_unique_excel(exl_file,sheet_name,cluster = "supertype.markers.combo",class_label = "class_label"):
    """
       Goal is to count unique genes per class type (only in one class) and record frequency (normalize by number of class clusters)

    """
    
    markers_df = pd.read_excel(exl_file,sheet_name=sheet_name)

    #interested in supertype.markers.combo and class_label and class_id_label
    markers_df = markers_df[[cluster,class_label]]
    markers_df = markers_df.dropna(subset=[cluster])
    #markers_df.to_csv("test.csv", index=False)

    markers_df['gene_list'] = markers_df[cluster].str.split(',').apply(
        lambda x: [g.strip() for g in x]
    )

    genes_long = (
        markers_df[[cluster, class_label]]
        .assign(gene=markers_df[cluster].str.split(','))
        .explode('gene')
    )

    # How many supertypes (rows) each gene appears in, per class
    gene_freq = (
        genes_long.groupby([class_label, 'gene'])
        .size()
        .rename('n_supertypes')          # frequency within class
        .reset_index()
    )

    class_totals = (
        markers_df.groupby(class_label)
        .size()
        .rename('n_total_supertypes')
        .reset_index()
    )


    # Merge into gene_freq
    gene_freq = gene_freq.merge(class_totals, on=class_label)
 
    # Compute frequency fraction per gene per class_type 
    gene_freq['freq_fraction'] = gene_freq['n_supertypes'] / gene_freq['n_total_supertypes']
    
    #gene_freq[gene_freq["gene"] == "1110015O18Rik"]
    #class_label	gene	n_supertypes	n_total_supertypes	freq_fraction
    #1583	MB Glut	1110015O18Rik	1	115	0.008696
    #2266	P GABA	1110015O18Rik	1	46	0.021739
    
    genes_long['gene'] = genes_long['gene'].str.strip()
    genes_long = genes_long[genes_long['gene'] != '']  # remove empty strings

    gene_class_count = (
        genes_long.groupby('gene')[class_label]
        .nunique()
        .rename('n_classes')
    )

    unique_genes = gene_class_count[gene_class_count == 1].index
    
    
    unique_table = (
        genes_long[genes_long['gene'].isin(unique_genes)]
        [[class_label, 'gene']]
        .drop_duplicates()
        .sort_values([class_label, 'gene'])
        .reset_index(drop=True)
    )

    #add frequency of gene out of total occurence of cluster
    unique_table = unique_table.merge(
        gene_freq[['gene', 'freq_fraction', 'n_supertypes', 'n_total_supertypes']],
        on='gene',
        how='left'
    )

    unique_marker_dict = (
        unique_table.groupby(class_label)['gene']
        .apply(list)
        .to_dict()
    )

    return markers_df,unique_table, unique_marker_dict,gene_freq,genes_long

In [3]:
markers_df,marker_table_allen,marker_allen,gene_freq,genes_long = marker_unique_excel('/research/groups/mackgrp/home/demerson/AIB/yao_et_al_Nature_2023.xlsx','supertype_annotation')
panel_5000_df = pd.read_excel("Xenium_Mouse_5000_Plex_Pan_Tissue_Panel_Gene_List_202403_CONFIDENTIAL.xlsx",sheet_name="Panel Targets 5006")


In [28]:
gene_freq

,class_label,gene,n_supertypes,n_total_supertypes,freq_fraction
0,Astro-Epen,2900040C04Rik,1,22,0.045455
1,Astro-Epen,A330076C08Rik,1,22,0.045455
2,Astro-Epen,Aqp4,1,22,0.045455
3,Astro-Epen,C230072F16Rik,1,22,0.045455
4,Astro-Epen,Camk2a,1,22,0.045455
...,...,...,...,...,...
2602,Vascular,Slc13a3,1,8,0.125000
2603,Vascular,Slc6a13,1,8,0.125000
2604,Vascular,Tfap2b,1,8,0.125000
2605,Vascular,Thbd,1,8,0.125000


In [37]:
class_genes = marker_table_allen.groupby('class_label')['gene'].unique().reset_index()
class_genes.columns = ['class_label', 'genes']


In [38]:
panel_genes = set(panel_5000_df['Gene'].unique())


In [51]:
# check which class genes are in the panel

class_genes['genes_in_panel'] = class_genes['genes'].apply(
    lambda x: [g for g in x if g in panel_genes]
)
class_genes['genes_in_panel_freq'] = class_genes['genes'].apply(
    lambda x: [list(gene_freq[gene_freq["gene"]== g]["freq_fraction"])[0] for g in x if g in panel_genes]
)
#for g in class_genes["genes"]:
#    for instance in g:
#        if instance in panel_genes:
#            print(instance)
#            print(gene_freq[gene_freq["gene"]== instance])

class_genes['n_genes_in_panel'] = class_genes['genes_in_panel'].apply(len)
class_genes['n_genes_total']    = class_genes['genes'].apply(len)
class_genes['pct_covered']      = (
    class_genes['n_genes_in_panel'] / class_genes['n_genes_total'] * 100
).round(1)

#class_genes['genes_present'] = class_genes['genes'].apply(
#    lambda x: [g for g in x if g in panel_genes]
#)

class_genes['genes_not_in_panel'] = class_genes['genes'].apply(
    lambda x: [g for g in x if g not in panel_genes]
)

class_genes['genes_not_in_panel_freq'] = class_genes['genes'].apply(
    lambda x: [list(gene_freq[gene_freq["gene"]== g]["freq_fraction"])[0] for g in x if g not in panel_genes]
)

def best_gene_idx(x):
    vals = [
        gene_freq.loc[gene_freq["gene"] == g, "freq_fraction"].iloc[0]
        for g in x
        if g not in panel_genes and (gene_freq["gene"] == g).any()
    ]
    return int(np.argmax(vals)) if len(vals) > 0 else np.nan

class_genes['max_id'] = class_genes["genes"].apply(best_gene_idx)

def pick_gene(r):
    genes = r["genes_not_in_panel"]
    idx = r["max_id"]
    if not isinstance(genes, list) or len(genes) == 0 or pd.isna(idx):
        return np.nan
    idx = int(idx)
    return genes[idx] if 0 <= idx < len(genes) else np.nan

class_genes["genes_not_in_panel_top_freq"] = class_genes.apply(pick_gene, axis=1)


#print(class_genes[['class_label','n_genes_total','n_genes_in_panel','pct_covered','genes_present','genes_missing']])


In [52]:
class_genes

,class_label,genes,genes_in_panel,genes_in_panel_freq,n_genes_in_panel,n_genes_total,pct_covered,genes_not_in_panel,genes_not_in_panel_freq,max_id,genes_not_in_panel_top_freq
0,Astro-Epen,"[2900040C04Rik, A330076C08Rik, Aqp4, C230072F1...","[Aqp4, Camk2a, Cnmd, Dao, Dcc, Ednrb, Fabp7, G...","[0.045454545454545456, 0.045454545454545456, 0...",18,34,52.9,"[2900040C04Rik, A330076C08Rik, C230072F16Rik, ...","[0.045454545454545456, 0.045454545454545456, 0...",9.0,Itih3
1,CB GABA,"[Afap1l2, Ddah2, Fat2, Gabrb1, Gldc, Il22, Ly6...","[Afap1l2, Gldc, Il22, Pcp2, Sdk2]","[0.25, 0.08333333333333333, 0.25, 0.0833333333...",5,9,55.6,"[Ddah2, Fat2, Gabrb1, Ly6d]","[0.08333333333333333, 0.08333333333333333, 0.0...",0.0,Ddah2
2,CB Glut,[Rab37],[],[],0,1,0.0,[Rab37],[0.3333333333333333],0.0,Rab37
3,CNU-HYa GABA,"[1700001F09Rik, A2ml1, Adora1, Ahi1, Aldh1a3, ...","[Adora1, Ahi1, Aldh1a3, Bcl6, Cdh6, Cdhr1, Cyp...","[0.010638297872340425, 0.010638297872340425, 0...",22,41,53.7,"[1700001F09Rik, A2ml1, Camk2n2, Col11a1, Csmd3...","[0.010638297872340425, 0.010638297872340425, 0...",10.0,Pgm5
4,CNU-HYa Glut,"[Car10, Cd36, Cngb3, Elf4, Fstl5, Gchfr, Glipr...","[Cd36, Cngb3, Fstl5, Glra3, Lama1, Lmo4, Rbfox...","[0.046511627906976744, 0.023255813953488372, 0...",9,24,37.5,"[Car10, Elf4, Gchfr, Glipr1, Iigp1, Il31ra, Kr...","[0.023255813953488372, 0.023255813953488372, 0...",11.0,Samd3
5,CNU-LGE GABA,"[1700003D09Rik, Acvrl1, Adgrd1, Adgrv1, Atp6ap...","[Acvrl1, Adgrv1, Bach2, Cd4, Cdh12, Efna5, Gpr...","[0.02857142857142857, 0.02857142857142857, 0.0...",13,30,43.3,"[1700003D09Rik, Adgrd1, Atp6ap1l, Cachd1, Carh...","[0.05714285714285714, 0.02857142857142857, 0.0...",0.0,1700003D09Rik
6,CNU-MGE GABA,"[Arhgap28, Avpr1a, Chodl, Colec12, Gdnf, Gpr17...","[Gdnf, Itga8, Sfrp1, Wnt2]","[0.07407407407407407, 0.037037037037037035, 0....",4,12,33.3,"[Arhgap28, Avpr1a, Chodl, Colec12, Gpr174, Par...","[0.037037037037037035, 0.037037037037037035, 0...",0.0,Arhgap28
7,CTX-CGE GABA,"[5330429C05Rik, 6530411M01Rik, Clrn1, Col26a1,...","[Col26a1, Grm8, Grpr, Nr1h4, Nrp2, Prkd3, Slit1]","[0.03333333333333333, 0.03333333333333333, 0.0...",7,18,38.9,"[5330429C05Rik, 6530411M01Rik, Clrn1, Frem2, K...","[0.03333333333333333, 0.03333333333333333, 0.0...",4.0,Krt73
8,CTX-MGE GABA,"[Adamts15, Adamtsl1, Anxa2, Atp2b4, C1qtnf7, C...","[Anxa2, Atp2b4, Cryab, Dmrt2, Egln3, Ets1, Hap...","[0.03333333333333333, 0.03333333333333333, 0.0...",10,26,38.5,"[Adamts15, Adamtsl1, C1qtnf7, Emilin2, Fam114a...","[0.03333333333333333, 0.03333333333333333, 0.0...",7.0,Gm31218
9,DG-IMN Glut,"[Atf3, Cntn6, Csrp2, Dsp, Lama2, Lct, Vwa3b]","[Cntn6, Dsp, Lama2]","[0.14285714285714285, 0.14285714285714285, 0.1...",3,7,42.9,"[Atf3, Csrp2, Lct, Vwa3b]","[0.14285714285714285, 0.14285714285714285, 0.1...",0.0,Atf3


In [33]:
#class_genes = class_genes.rename(columns={'genes': 'allen_total_genes'})
#class_genes = class_genes.rename(columns={'genes_missing': 'genes_not_in_panel'})

In [53]:
class_genes.to_excel("allen_class_Xenium_Mouse_5000_Plex_Pan_Tissue_Panel_Gene_List_202403_coverage.xlsx", index=False)

In [35]:
class_genes

,class_label,allen_total_genes,genes_in_panel,genes_in_panel_freq,n_genes_in_panel,n_genes_total,pct_covered,genes_not_in_panel,genes_not_in_panel_freq
0,Astro-Epen,"[2900040C04Rik, A330076C08Rik, Aqp4, C230072F1...","[Aqp4, Camk2a, Cnmd, Dao, Dcc, Ednrb, Fabp7, G...","[0.045454545454545456, 0.045454545454545456, 0...",18,34,52.9,"[2900040C04Rik, A330076C08Rik, C230072F16Rik, ...","[0.045454545454545456, 0.045454545454545456, 0..."
1,CB GABA,"[Afap1l2, Ddah2, Fat2, Gabrb1, Gldc, Il22, Ly6...","[Afap1l2, Gldc, Il22, Pcp2, Sdk2]","[0.25, 0.08333333333333333, 0.25, 0.0833333333...",5,9,55.6,"[Ddah2, Fat2, Gabrb1, Ly6d]","[0.08333333333333333, 0.08333333333333333, 0.0..."
2,CB Glut,[Rab37],[],[],0,1,0.0,[Rab37],[0.3333333333333333]
3,CNU-HYa GABA,"[1700001F09Rik, A2ml1, Adora1, Ahi1, Aldh1a3, ...","[Adora1, Ahi1, Aldh1a3, Bcl6, Cdh6, Cdhr1, Cyp...","[0.010638297872340425, 0.010638297872340425, 0...",22,41,53.7,"[1700001F09Rik, A2ml1, Camk2n2, Col11a1, Csmd3...","[0.010638297872340425, 0.010638297872340425, 0..."
4,CNU-HYa Glut,"[Car10, Cd36, Cngb3, Elf4, Fstl5, Gchfr, Glipr...","[Cd36, Cngb3, Fstl5, Glra3, Lama1, Lmo4, Rbfox...","[0.046511627906976744, 0.023255813953488372, 0...",9,24,37.5,"[Car10, Elf4, Gchfr, Glipr1, Iigp1, Il31ra, Kr...","[0.023255813953488372, 0.023255813953488372, 0..."
5,CNU-LGE GABA,"[1700003D09Rik, Acvrl1, Adgrd1, Adgrv1, Atp6ap...","[Acvrl1, Adgrv1, Bach2, Cd4, Cdh12, Efna5, Gpr...","[0.02857142857142857, 0.02857142857142857, 0.0...",13,30,43.3,"[1700003D09Rik, Adgrd1, Atp6ap1l, Cachd1, Carh...","[0.05714285714285714, 0.02857142857142857, 0.0..."
6,CNU-MGE GABA,"[Arhgap28, Avpr1a, Chodl, Colec12, Gdnf, Gpr17...","[Gdnf, Itga8, Sfrp1, Wnt2]","[0.07407407407407407, 0.037037037037037035, 0....",4,12,33.3,"[Arhgap28, Avpr1a, Chodl, Colec12, Gpr174, Par...","[0.037037037037037035, 0.037037037037037035, 0..."
7,CTX-CGE GABA,"[5330429C05Rik, 6530411M01Rik, Clrn1, Col26a1,...","[Col26a1, Grm8, Grpr, Nr1h4, Nrp2, Prkd3, Slit1]","[0.03333333333333333, 0.03333333333333333, 0.0...",7,18,38.9,"[5330429C05Rik, 6530411M01Rik, Clrn1, Frem2, K...","[0.03333333333333333, 0.03333333333333333, 0.0..."
8,CTX-MGE GABA,"[Adamts15, Adamtsl1, Anxa2, Atp2b4, C1qtnf7, C...","[Anxa2, Atp2b4, Cryab, Dmrt2, Egln3, Ets1, Hap...","[0.03333333333333333, 0.03333333333333333, 0.0...",10,26,38.5,"[Adamts15, Adamtsl1, C1qtnf7, Emilin2, Fam114a...","[0.03333333333333333, 0.03333333333333333, 0.0..."
9,DG-IMN Glut,"[Atf3, Cntn6, Csrp2, Dsp, Lama2, Lct, Vwa3b]","[Cntn6, Dsp, Lama2]","[0.14285714285714285, 0.14285714285714285, 0.1...",3,7,42.9,"[Atf3, Csrp2, Lct, Vwa3b]","[0.14285714285714285, 0.14285714285714285, 0.1..."


In [28]:
genes_long

,supertype.markers.combo,class_label,gene
0,"Car3,Nrg3,Gm34567",IT-ET Glut,Car3
0,"Car3,Nrg3,Gm34567",IT-ET Glut,Nrg3
0,"Car3,Nrg3,Gm34567",IT-ET Glut,Gm34567
1,"Ccnb1,Tcf7l1",IT-ET Glut,Ccnb1
1,"Ccnb1,Tcf7l1",IT-ET Glut,Tcf7l1
...,...,...,...
1198,"Il1rl1,Gata3",Immune,Gata3
1199,"Ccl5,Fcer1g",Immune,Ccl5
1199,"Ccl5,Fcer1g",Immune,Fcer1g
1200,"Itk,Cd3d",Immune,Itk


In [67]:
gene_freq[gene_freq["gene"]=="Nrg3"]

,class_label,gene,n_supertypes,n_total_supertypes,freq_fraction
342,CNU-HYa Glut,Nrg3,1,43,0.023256
1198,IT-ET Glut,Nrg3,1,102,0.009804
2447,P Glut,Nrg3,1,56,0.017857


In [70]:
marker_table_allen

,class_label,gene
0,Astro-Epen,2900040C04Rik
1,Astro-Epen,A330076C08Rik
2,Astro-Epen,Aqp4
3,Astro-Epen,C230072F16Rik
4,Astro-Epen,Camk2a
...,...,...
747,Vascular,Myh11
748,Vascular,Slc13a3
749,Vascular,Slc6a13
750,Vascular,Thbd


In [71]:
gene_freq[gene_freq['gene']=='2900040C04Rik']

,class_label,gene,n_supertypes,n_total_supertypes,freq_fraction
0,Astro-Epen,2900040C04Rik,1,22,0.045455


In [75]:
gene_freq[gene_freq['gene']=='1110015O18Rik']

,class_label,gene,n_supertypes,n_total_supertypes,freq_fraction
1583,MB Glut,1110015O18Rik,1,115,0.008696
2266,P GABA,1110015O18Rik,1,46,0.021739


In [77]:
marker_table_allen = marker_table_allen.merge(
    gene_freq[['gene', 'freq_fraction', 'n_supertypes', 'n_total_supertypes']],
    on='gene',
    how='left'
)

In [81]:
marker_table_allen

,class_label,gene,freq_fraction,n_supertypes,n_total_supertypes
0,Astro-Epen,2900040C04Rik,0.045455,1,22
1,Astro-Epen,A330076C08Rik,0.045455,1,22
2,Astro-Epen,Aqp4,0.045455,1,22
3,Astro-Epen,C230072F16Rik,0.045455,1,22
4,Astro-Epen,Camk2a,0.045455,1,22
...,...,...,...,...,...
747,Vascular,Myh11,0.125000,1,8
748,Vascular,Slc13a3,0.125000,1,8
749,Vascular,Slc6a13,0.125000,1,8
750,Vascular,Thbd,0.125000,1,8


In [83]:
marker_table_allen

,class_label,gene,freq_fraction,n_supertypes,n_total_supertypes
0,Astro-Epen,2900040C04Rik,0.045455,1,22
1,Astro-Epen,A330076C08Rik,0.045455,1,22
2,Astro-Epen,Aqp4,0.045455,1,22
3,Astro-Epen,C230072F16Rik,0.045455,1,22
4,Astro-Epen,Camk2a,0.045455,1,22
...,...,...,...,...,...
747,Vascular,Myh11,0.125000,1,8
748,Vascular,Slc13a3,0.125000,1,8
749,Vascular,Slc6a13,0.125000,1,8
750,Vascular,Thbd,0.125000,1,8
